## Objetivo e perguntas

A base Gapminder reúne dados reais de expectativa de vida, PIB per capita e população para diversos países, de 1952 a 2007. É a mesma base usada por Hans Rosling em suas famosas apresentações sobre desenvolvimento global.

Perguntas que guiam esta análise:

1. Como a expectativa de vida evoluiu no mundo ao longo do tempo?
2. Existe relação entre PIB per capita e expectativa de vida? É linear?
3. Os continentes têm distribuições de expectativa de vida muito diferentes entre si?
4. Quais países mais progrediram (ou regrediram) entre 1952 e 2007?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats

sns.set_theme(style="whitegrid")

df = px.data.gapminder()
df.head()

## 1. Estrutura dos dados

In [ ]:
print(f"Formato: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.dtypes

In [ ]:
df.info()

## 2. Qualidade dos dados

Verificando valores ausentes, duplicados e a cobertura temporal/geográfica.

In [ ]:
print("Valores ausentes por coluna:")
print(df.isna().sum())
print(f"\nLinhas duplicadas: {df.duplicated().sum()}")
print(f"\nAnos cobertos: {sorted(df['year'].unique())}")
print(f"Continentes: {df['continent'].unique().tolist()}")
print(f"Número de países: {df['country'].nunique()}")

Base limpa: sem valores ausentes nem duplicados, com 142 países observados a cada 5 anos entre 1952 e 2007 — um painel bem balanceado.

## 3. Estatística descritiva univariada

### Variáveis numéricas

In [ ]:
df[["lifeExp", "pop", "gdpPercap"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df["lifeExp"], kde=True, ax=axes[0])
axes[0].set_title("Expectativa de vida")

sns.histplot(df["gdpPercap"], kde=True, ax=axes[1])
axes[1].set_title("PIB per capita")

sns.histplot(np.log10(df["pop"]), kde=True, ax=axes[2])
axes[2].set_title("População (escala log10)")

plt.tight_layout()
plt.show()

`gdpPercap` e `pop` são fortemente assimétricas à direita (poucos países muito ricos/populosos puxam a cauda) — por isso a população é melhor visualizada em escala logarítmica. `lifeExp` já se aproxima mais de uma distribuição simétrica, com uma leve concentração à esquerda (países com expectativa de vida baixa).

### Variável categórica: continente

In [ ]:
df["continent"].value_counts()

## 4. Outliers

Olhando os extremos de `gdpPercap` via regra do IQR.

In [ ]:
q1, q3 = df["gdpPercap"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

outliers = df[df["gdpPercap"] > limite_superior]
print(f"Limite superior (IQR): {limite_superior:,.0f}")
print(f"Número de observações acima do limite: {len(outliers)}")
outliers.sort_values("gdpPercap", ascending=False)[["country", "year", "gdpPercap"]].head(10)

Os "outliers" aqui não são erros — são países genuinamente muito ricos (ex: Kuwait em décadas de petróleo caro). Isso reforça que outlier estatístico não é sinônimo de dado errado: é preciso investigar antes de remover.

## 5. Pergunta 1 — Evolução da expectativa de vida no mundo

In [ ]:
evolucao = df.groupby("year")["lifeExp"].mean()

plt.figure(figsize=(9, 4))
evolucao.plot(marker="o")
plt.title("Expectativa de vida média mundial ao longo do tempo")
plt.ylabel("Expectativa de vida (anos)")
plt.xlabel("Ano")
plt.show()

A expectativa de vida média mundial subiu de forma quase monotônica entre 1952 e 2007, com uma única queda visível em 1992 (associada a crises e epidemias em partes da África e ex-URSS).

## 6. Pergunta 2 — Relação entre PIB per capita e expectativa de vida

In [ ]:
df_2007 = df[df["year"] == 2007]

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df_2007, x="gdpPercap", y="lifeExp", hue="continent", size="pop",
    sizes=(20, 400), alpha=0.7
)
plt.xscale("log")
plt.title("PIB per capita x Expectativa de vida (2007)")
plt.xlabel("PIB per capita (escala log)")
plt.ylabel("Expectativa de vida")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
corr_linear = df_2007["gdpPercap"].corr(df_2007["lifeExp"])
corr_log = np.log(df_2007["gdpPercap"]).corr(df_2007["lifeExp"])

print(f"Correlação (PIB per capita, expectativa de vida): {corr_linear:.2f}")
print(f"Correlação (log do PIB per capita, expectativa de vida): {corr_log:.2f}")

A relação não é linear em escala bruta, mas fica muito mais forte quando usamos o log do PIB per capita — um padrão clássico em economia: ganhos de expectativa de vida são grandes nos primeiros aumentos de renda e desaceleram (retornos marginais decrescentes) em países já ricos.

## 7. Pergunta 3 — Expectativa de vida difere entre continentes?

In [ ]:
plt.figure(figsize=(8, 5))
ordem = df_2007.groupby("continent")["lifeExp"].median().sort_values().index
sns.boxplot(data=df_2007, x="continent", y="lifeExp", order=ordem)
plt.title("Expectativa de vida por continente (2007)")
plt.show()

In [ ]:
grupos = [g["lifeExp"].values for _, g in df_2007.groupby("continent")]
f_stat, p_valor = stats.f_oneway(*grupos)
print(f"ANOVA — estatística F: {f_stat:.2f}, p-valor: {p_valor:.2e}")

O p-valor extremamente baixo confirma o que o boxplot já sugere visualmente: as médias de expectativa de vida diferem significativamente entre continentes. A África tem a menor mediana e a maior dispersão, enquanto Oceania e Europa têm as maiores medianas e menor variabilidade.

## 8. Pergunta 4 — Países que mais progrediram (ou regrediram)

In [ ]:
pivot = df[df["year"].isin([1952, 2007])].pivot(
    index="country", columns="year", values="lifeExp"
)
pivot["variacao"] = pivot[2007] - pivot[1952]

print("Top 5 maiores ganhos de expectativa de vida (1952 -> 2007):")
print(pivot.sort_values("variacao", ascending=False).head(5))

print("\nTop 5 maiores quedas de expectativa de vida (1952 -> 2007):")
print(pivot.sort_values("variacao").head(5))

Os maiores avanços vêm de países asiáticos que passaram por forte desenvolvimento econômico e de saúde pública no período (ex: melhorias sanitárias e redução de mortalidade infantil). As quedas concentram-se em países africanos duramente afetados pela epidemia de HIV/AIDS nas décadas de 1990-2000.

## Conclusões

- A expectativa de vida mundial cresceu de forma consistente entre 1952 e 2007, com uma queda pontual em 1992.
- PIB per capita e expectativa de vida têm relação forte, porém não linear: log(PIB per capita) é um preditor muito melhor, evidenciando retornos marginais decrescentes de renda sobre saúde.
- Continentes diferem estatisticamente (ANOVA, p < 0.001) na expectativa de vida — África com menor mediana e maior dispersão, Europa e Oceania com maior mediana e menor dispersão.
- As maiores mudanças (positivas e negativas) ao longo do período estão ligadas a eventos históricos concretos: desenvolvimento acelerado na Ásia e a epidemia de HIV/AIDS na África.

Próximos passos possíveis: usar `gdpPercap` (em log), `continent` e `pop` para treinar um modelo de regressão que preveja `lifeExp`, e comparar o poder explicativo desses fatores.